# Exercise 02 — the "star operation", built on the engine

## Your task

Build a few small layers as `nn.Module` subclasses that use the **star operation** —
element-wise multiplication of two learned affine branches — and compare it with the usual
additive form. You implement only `forward`: because each layer is composed out of ops the
engine already differentiates (`@`, `+`, `*`, `relu`, ...), **the engine gives you the
backward pass for free.**

This is the key contrast with [Exercise 01](q01_activations.ipynb): there you added a
*brand-new* op and so had to hand-write its `_backward` (the local derivative). Here you
add *no* new op — you only compose existing ones — so there is no backward to write. The
autograd graph (each result tensor remembering its operands in `_prev`) takes care of it.
The gradient check at the bottom proves it: the analytic gradients from `backward()` match
finite differences, even though you never wrote a derivative.

## The maths

A traditional two-branch layer *adds* the branches:

$$u = W_1^{\top}\,[1; x] \quad\text{(branch 1, the bias folded into row 0)}$$
$$v = W_2^{\top}\,[1; x] \quad\text{(branch 2)}$$
$$y = \mathrm{act}(u) + v \qquad \text{(SumLinear)}$$

The "star" form *multiplies* them element-wise:

$$y = \mathrm{act}(u) \odot v \qquad \text{(StarLinear)}$$

Expanding one output unit [1], $(w_1^{\top}x + b_1)(w_2^{\top}x + b_2)$ contains terms
$x_i x_j$, $x_i$ and constants — so the star layer represents implicit second-order feature
interactions without ever building the $x_i x_j$ pairs.

## Required reading

[1] Ma, X., Dai, X., Bai, Y., Wang, Y., & Fu, Y. (2024). *Rewrite the Stars.*
CVPR 2024, pp. 5694-5703.

## Conventions

Like the rest of the library, inputs are **column-oriented**: `x` has shape
`(in_features, batch)` and each `nn.Linear` computes `Wᵀ @ [1; x]` (the bias trick). So
`u`, `v` and `y` have shape `(out_features, batch)` and `*` multiplies them element-wise.

## How to work

1. Run the setup cell.
2. Fill in the four `forward` methods (remove each `raise NotImplementedError`), running
   each cell as you go.
3. Run the **grading** cell: it gradient-checks your layers against finite differences.
4. Run the **second-order** cell to see what the star op buys you.

> **Tip:** a layer is just another graph. Look at `bert_cpu/nn.py::Linear` — its `forward`
> builds the output from engine ops and returns it; nothing else. Yours do the same, just
> combining two branches with `+` or `*`.

In [ ]:
# Run me first: make ``bert_cpu`` importable whether Jupyter was started in the
# project root or inside ``exercises/``, exactly like the scripts in this folder do.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from typing import Callable

import numpy as np

from bert_cpu import engine as cpu
from bert_cpu import nn
from exercises.grading import gradient_check, report

# An activation is just a function Tensor -> Tensor; ReLU by default. Pass
# ``lambda t: t`` for the identity (handy for the gradient check).
Activation = Callable[[cpu.Tensor], cpu.Tensor]
_RELU: Activation = lambda t: t.relu()
_IDENTITY: Activation = lambda t: t

print("ready — engine imported from", ROOT)

## Exercise 1 — additive two-branch layer

`SumLinear`: two affine branches, combined by addition,
`y = act(W1ᵀ @ [1;x]) + W2ᵀ @ [1;x]`.

The two `nn.Linear` branches are stored as attributes, so `Module` collects their
parameters automatically (`parameters()` returns both weights).

**Questions to ponder**

1. With the identity activation, can the two branches be collapsed into one `Linear`?
   (What does that say about additive fusion?)
2. What does ReLU on one branch buy you?

In [ ]:
class SumLinear(nn.Module):
    """Two affine branches, combined by addition."""

    def __init__(self, in_features: int, out_features: int, activation: Activation = _RELU) -> None:
        self.activation = activation
        self.branch1 = nn.Linear(in_features, out_features)
        self.branch2 = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement SumLinear.forward")

## Exercise 2 — the star-operation layer

`StarLinear`: two affine branches, combined by element-wise product (the star op),

`y = act(W1ᵀ @ [1;x]) * (W2ᵀ @ [1;x])`

You do not implement a backward pass. `*` (`Tensor.__mul__`) already carries the product
rule, `@` carries the matmul rule, and so on — so the engine differentiates `y` for you.
The product rule it applies is exactly `dL/da = dL/dy * v` and `dL/dv = dL/dy * a`
(with `a = act(u)`).

In [ ]:
class StarLinear(nn.Module):
    """Two affine branches, combined by element-wise product (the star op)."""

    def __init__(self, in_features: int, out_features: int, activation: Activation = _RELU) -> None:
        self.activation = activation
        self.branch1 = nn.Linear(in_features, out_features)
        self.branch2 = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement StarLinear.forward")

## Exercise 3 — the "identity branch" special case

`IdentityStarLinear`: one transformed branch times the input itself,
`y = act(Wᵀ @ [1;x]) * x`.

Requires `out_features == in_features` so the shapes line up for `*`. Note that `x` reaches
`y` through *two* paths (directly, and through the transformed branch); the engine sums
both contributions into `x.grad` automatically — you do not manage that.

In [ ]:
class IdentityStarLinear(nn.Module):
    """One transformed branch times the input itself."""

    def __init__(self, features: int, activation: Activation = _RELU) -> None:
        self.activation = activation
        self.linear = nn.Linear(features, features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement IdentityStarLinear.forward")

## Exercise 4 — the simplest star: square the features

`SquareFeature`: the simplest star case, `y = x * x` — no parameters, but non-linear.

With `parameters()` empty, this layer is pure structure; it still produces a differentiable
node (`x * x`) whose gradient `2x` the engine derives.

In [ ]:
class SquareFeature(nn.Module):
    """The simplest star case, ``y = x * x``."""

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement SquareFeature.forward")

## GIVEN — the grading harness

You do not edit the cells below.

`gradient_check` (in [`exercises/grading.py`](grading.py), shared by every exercise here)
builds `loss = mean((layer(x) - target)**2)`, runs `backward()`, and then recomputes each
gradient with **central finite differences** — perturbing one number at a time and
re-running the forward pass. Those finite differences know nothing about the engine: no
`_prev`, no `_backward`, no chain rule. That is why agreement is real evidence, and why it
is worth pausing on: you wrote four forwards and zero derivatives, and the derivatives are
right.

In [ ]:
def grade() -> None:
    """Gradient-check every layer (identity activation keeps it smooth)."""
    cpu.set_seed(0)
    n_in, n_out, batch = 3, 2, 4
    x = cpu.Tensor(np.random.randn(n_in, batch))
    y2 = cpu.Tensor(np.random.randn(n_out, batch))
    yk = cpu.Tensor(np.random.randn(n_in, batch))   # square-shaped target

    cases = [
        ("SumLinear",         SumLinear(n_in, n_out, activation=_IDENTITY),  x, y2),
        ("StarLinear",        StarLinear(n_in, n_out, activation=_IDENTITY), x, y2),
        ("IdentityStarLinear", IdentityStarLinear(n_in, activation=_IDENTITY), x, yk),
        ("SquareFeature",     SquareFeature(),                                x, yk),
    ]
    print("Gradient check (analytic backward vs finite differences):\n")
    all_ok = True
    for name, layer, xin, target in cases:
        print(f"  {name}:")
        try:
            worst = gradient_check(layer, xin, target)
        except NotImplementedError as exc:
            all_ok = False
            print(f"    -> SKIPPED ({exc}).\n")
            continue
        ok = worst < 1e-5
        all_ok = all_ok and ok
        print(f"    -> {'PASS' if ok else 'FAIL'} (worst {worst:.2e})\n")
    report(all_ok, "layers")

In [ ]:
try:
    grade()
except NotImplementedError as exc:
    print(f"Implement the forwards first ({exc}).")

## The star op is second-order

With the identity activation and a single output, `StarLinear` computes
$(w_1^{\top}x + b_1)(w_2^{\top}x + b_2)$. The cell below freezes a tiny example and reads
off the mixed second-order coefficient with a second finite difference,
$\partial^2 y / \partial x_0 \partial x_1$ — non-zero for the star op, exactly zero for the
sum op. That single number is the whole point of the paper.

In [ ]:
def demonstrate_second_order() -> None:
    """Show the star op carries an ``x_i x_j`` term that the sum op cannot."""
    cpu.set_seed(1)
    star = StarLinear(2, 1, activation=_IDENTITY)
    summ = SumLinear(2, 1, activation=_IDENTITY)

    def mixed_second_derivative(layer: nn.Module, h: float = 1e-3) -> float:
        def f(a: float, b: float) -> float:
            xv = cpu.Tensor(np.array([[a], [b]], dtype=float))
            return float(layer(xv).data.reshape(()))
        # d2/da db via the standard 4-point stencil.
        return (f(h, h) - f(h, -h) - f(-h, h) + f(-h, -h)) / (4 * h * h)

    print("Second-order interaction  d2y / dx0 dx1  (identity activation):")
    print(f"    StarLinear : {mixed_second_derivative(star):+.4f}   (non-zero -> learns x0*x1)")
    print(f"    SumLinear  : {mixed_second_derivative(summ):+.4f}   (zero      -> stays additive)")


try:
    demonstrate_second_order()
except NotImplementedError as exc:
    print(f"Implement the forwards first ({exc}).")

---

**Next:** [Exercise 03 — Gated Linear Units](q03_gated_linear_units.ipynb). Same star
shape, but one branch is squashed into `(0, 1)` so it acts as a *gate* — and the SwiGLU
there gates with the `swish` you hand-wrote in Exercise 01.